# Mobile Money Agent Network: Data Audit and Cleaning

This notebook audits the supplied synthetic transaction extract, calls the reusable `clean_agent_data()` function from `cleaning_pipeline.py`, validates the cleaned output and produces operational summaries.

## 1. Load and inspect source files

In [1]:
import pandas as pd
import numpy as np
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
print("Raw transactions:", txns.shape)
print("Agent register:", agents.shape)
display(txns.head())
display(txns.isna().sum().to_frame("missing_values"))
display(txns["txn_type"].value_counts(dropna=False).to_frame("rows"))
display(txns["amount"].value_counts(dropna=False).head(10).to_frame("rows"))

Raw transactions: (612, 5)
Agent register: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


,missing_values
txn_id,0
agent_id,0
txn_type,0
amount,22
txn_date,0


,rows
txn_type,
deposit,142
withdrawal,131
Deposit,128
DEPOSIT,111
Withdrawal,100


,rows
amount,
NaN,22
250000.0,5
1455.0,3
893.0,3
2501.0,3
2383.0,3
1062.0,2
2678.0,2
1007.0,2


## 2. Data-quality audit

The audit below records detected issues, the quality dimension affected, the relevant column and the number of affected rows. `N/A` is interpreted as missing by pandas' default CSV parser, so missingness and failed numeric conversion are checked explicitly.

In [2]:
raw = txns.copy()
amount_numeric = pd.to_numeric(raw["amount"], errors="coerce")
type_normalised = raw["txn_type"].astype("string").str.strip().str.casefold()
audit = pd.DataFrame([
    {"issue": "Duplicate transaction IDs", "dimension": "Uniqueness", "column": "txn_id",
     "rows_affected": int(raw.duplicated("txn_id").sum())},
    {"issue": "Missing or non-numeric amounts (including N/A)", "dimension": "Completeness/Validity", "column": "amount",
     "rows_affected": int(amount_numeric.isna().sum())},
    {"issue": "Transaction-type formatting inconsistent with canonical labels", "dimension": "Consistency", "column": "txn_type",
     "rows_affected": int((raw["txn_type"].astype(str) != type_normalised.map({"deposit":"Deposit","withdrawal":"Withdrawal"}).astype(str)).sum())},
    {"issue": "Slash-formatted dates", "dimension": "Consistency", "column": "txn_date",
     "rows_affected": int(raw["txn_date"].astype(str).str.contains("/", regex=False).sum())},
    {"issue": "Unknown agent IDs", "dimension": "Completeness", "column": "agent_id",
     "rows_affected": int((~raw["agent_id"].isin(agents["agent_id"])).sum())},
    {"issue": "KES 250,000 values requiring verification", "dimension": "Accuracy", "column": "amount",
     "rows_affected": int((amount_numeric == 250000).sum())},
])
display(audit)

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing or non-numeric amounts (including N/A),Completeness/Validity,amount,22
2,Transaction-type formatting inconsistent with ...,Consistency,txn_type,384
3,Slash-formatted dates,Consistency,txn_date,5
4,Unknown agent IDs,Completeness,agent_id,8
5,"KES 250,000 values requiring verification",Accuracy,amount,5


## 3. Apply the reusable cleaning pipeline

The function removes repeated `txn_id` values (keeping the first record), standardises transaction types, converts amounts to numeric, imputes missing amounts using the median within transaction type, flags IQR outliers without deleting them, parses mixed date formats and left-joins the agent register.

In [3]:
clean_df, cleaning_log = clean_agent_data(txns, agents)
display(cleaning_log)
print("Rows before cleaning:", len(txns))
print("Rows after deduplication and cleaning:", len(clean_df))
print("Duplicate txn_id remaining:", clean_df["txn_id"].duplicated().sum())
print("Transaction types:", sorted(clean_df["txn_type"].dropna().unique()))
print("Unknown agent transactions:", int((clean_df["county"] == "Unknown").sum()))
print("Missing amount values remaining:", clean_df["amount"].isna().sum())
print("Missing dates remaining:", clean_df["txn_date"].isna().sum())

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs removed,Uniqueness,txn_id,12
1,Missing or unrecognised transaction type,Consistency,txn_type,0
2,Missing or non-numeric amount values,Completeness/Validity,amount,20
3,IQR-flagged suspicious amounts,Validity/Accuracy,amount,53
4,"KES 250,000 values retained for verification",Accuracy,amount,5
5,Unparseable transaction dates,Validity,txn_date,0
6,Transactions linked to unknown agent IDs,Completeness,agent_id/county,7


Rows before cleaning: 612
Rows after deduplication and cleaning: 600
Duplicate txn_id remaining: 0
Transaction types: ['Deposit', 'Withdrawal']
Unknown agent transactions: 7
Missing amount values remaining: 0
Missing dates remaining: 0


## 4. Cleaning decisions and rationale

- **Duplicates:** `txn_id` is treated as the unique transaction key; duplicate IDs are removed by keeping the first occurrence.
- **Transaction type:** whitespace and case are normalised into exactly `Deposit` and `Withdrawal`. Unexpected future labels are retained as `Unknown` for investigation rather than silently misclassified.
- **Missing/invalid amounts:** numeric coercion converts blank and `N/A` values to missing. Missing amounts are imputed using the median of the corresponding transaction type, with the overall median as a fallback. `amount_imputed` preserves an audit flag; imputed values should not be treated as observed cash amounts in financial reconciliation.
- **Outliers:** the IQR method flags unusually low/high amounts in `is_outlier` but does not automatically delete them. The injected KES 250,000 values are retained and flagged/reviewed against transaction receipts, limits and agent records before any reversal or exclusion.
- **Dates:** ISO and slash-separated dates are parsed into datetime values. The left join preserves transactions even where the agent ID is not in the register; those counties are labelled `Unknown` pending register reconciliation.

## 5. Outlier investigation

In [4]:
q1 = clean_df["amount"].quantile(0.25)
q3 = clean_df["amount"].quantile(0.75)
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR lower bound: KES {lower_bound:,.2f}")
print(f"IQR upper bound: KES {upper_bound:,.2f}")
print("IQR-flagged transactions:", int(clean_df["is_outlier"].sum()))
display(clean_df.loc[(clean_df["amount"] == 250000),
                     ["txn_id","agent_id","txn_type","amount","txn_date","is_outlier","county"]])

IQR lower bound: KES -1,540.25
IQR upper bound: KES 5,513.75
IQR-flagged transactions: 53


,txn_id,agent_id,txn_type,amount,txn_date,is_outlier,county
14,TX00307,AG012,Deposit,250000.0,2026-03-02,True,Kisumu
17,TX00425,AG005,Withdrawal,250000.0,2026-03-02,True,Mombasa
120,TX00595,AG016,Deposit,250000.0,2026-03-11,True,Kiambu
303,TX00316,AG014,Deposit,250000.0,2026-04-01,True,Nairobi
389,TX00464,AG009,Withdrawal,250000.0,2026-04-11,True,Nakuru


## 6. County summary by transaction type

In [5]:
county_summary = (clean_df.groupby(["county", "txn_type"], dropna=False)
                  .agg(total_value=("amount","sum"),
                       transactions=("txn_id","count"),
                       average_value=("amount","mean"))
                  .reset_index()
                  .sort_values(["county","txn_type"]))
display(county_summary)

,county,txn_type,total_value,transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77024.5,27,2852.759259
2,Kisumu,Deposit,436501.5,74,5898.668919
3,Kisumu,Withdrawal,76348.5,37,2063.472973
4,Mombasa,Deposit,149094.0,72,2070.750000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543698.0,118,4607.610169
7,Nairobi,Withdrawal,161932.0,62,2611.806452
8,Nakuru,Deposit,184944.0,60,3082.400000
9,Nakuru,Withdrawal,328568.5,39,8424.833333


## 7. Weekly transaction-value pivot by county

In [6]:
clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)
weekly_pivot = pd.pivot_table(clean_df, index="week", columns="county",
                              values="amount", aggfunc="sum", fill_value=0)
display(weekly_pivot)

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week,,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2770.5,0.0
2026-03-02/2026-03-08,18317.0,287933.0,285696.5,68967.0,56574.0,0.0
2026-03-09/2026-03-15,267745.5,23186.0,30504.5,52887.5,28730.0,0.0
2026-03-16/2026-03-22,22054.0,28126.0,25171.5,43134.5,39795.0,1639.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41400.5,18972.5,0.0
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06/2026-04-12,15030.0,15341.0,47119.5,56813.5,276609.0,9824.0
2026-04-13/2026-04-19,12265.0,29470.0,26730.0,56406.5,26179.5,0.0
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65135.0,30862.0,1877.0


## 8. Export cleaned data

In [7]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
export_df = clean_df.drop(columns=["week"], errors="ignore").copy()
export_df["txn_date"] = export_df["txn_date"].dt.strftime("%Y-%m-%d")
export_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Exported outputs/agent_transactions_clean.csv")
print("Columns:", export_df.columns.tolist())

Exported outputs/agent_transactions_clean.csv
Columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county']


## 9. Data Quality Report

### Completeness
Missing and non-numeric transaction amounts were identified and imputed with the transaction-type median. The `amount_imputed` flag distinguishes estimated values from observed values. Unknown agent IDs are retained and assigned county `Unknown` until the agent register is corrected.

### Uniqueness
Duplicate transactions were detected using `txn_id` and removed by retaining the first occurrence. The transaction ID is the business key used to prevent duplicate counting.

### Validity
Amounts were converted to numeric values; dates were parsed into datetime values. The IQR method was used to flag unusual amounts. High-value KES 250,000 entries should be verified against source receipts and business rules, not automatically deleted.

### Consistency
Transaction types were standardised to `Deposit` and `Withdrawal`. Slash-formatted dates were parsed consistently and exported in ISO `YYYY-MM-DD` format.

### Accuracy
Automated checks can identify implausible values but cannot prove that a transaction amount is factually correct. Outlier flags and source-document verification are recommended before changing or reversing high-value transactions.

### Timeliness
The extract covers a defined period in March and April 2026. Operational reporting should include an extraction timestamp and a routine refresh schedule so decisions are based on current transactions.

### Privacy and the Kenya Data Protection Act, 2019
If customer phone numbers were included, they would be personal data because they can identify or make a customer identifiable. The organisation should establish a lawful purpose and basis for processing, give clear privacy information, collect only necessary fields, restrict access by role, encrypt data in transit and at rest, mask or pseudonymise phone numbers in analytical extracts, set retention periods, securely delete data when no longer needed, and apply appropriate safeguards to processors. A data-protection impact assessment may be required for high-risk processing, and personal-data breaches should be handled and notified in accordance with the Act and applicable ODPC requirements.